In [1]:
# Import the packages
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:

# Configuration and Feature Definitions

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu") #Automatically selects GPU acceleration (cuda) if available; otherwise falls back to cpu
BATCH_SIZE = 256 #Feeds data into the model in mini-batches of 256 samples.
EPOCHS = 100 # Defines the maximum training iterations over the dataset
PATIENCE = 15 # Sets early stopping threshold to stop training if validation performance stops improving for 15 consecutive epochs.
LEARNING_RATE = 1e-3 #Controls the optimizer step size ($0.001$)

SELECTED_FEATURES = [
    "m (kg)",
    "ec (cm3)",
    "ep (KW)",
    "Ft_diesel",
    "Ft_petrol",
    "Ft_petrol/electric",
    "Ft_diesel/electric",
    "Fm_H",
    "Fm_M",
    "Fm_P",
    "Electric range (km)",
    "IT_28",
    "IT_29",
    "IT_32",
    "IT_33",
    "IT_37",
    "IT_38",
    "IT_none",
]

In [3]:

# PyTorch Dataset Definition

class VehicleDataset(Dataset):
    def __init__(self, X_df: pd.DataFrame, y_df: pd.DataFrame):
        self.X = torch.tensor(X_df[SELECTED_FEATURES].values, dtype=torch.float32)
        self.y = torch.tensor(y_df.values, dtype=torch.float32).view(-1, 1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [4]:
# This data is taken from the "Export Encoded and Scaled data" cell in 03.5_final_remodelling
from pathlib import Path

split_dir = Path("../data/processed/eu/splits/final")

X_train_df = pd.read_parquet(split_dir / "X_train_encoded_scaled_final.parquet")
y_train_df = pd.read_parquet(split_dir / "y_train.parquet")

X_val_df = pd.read_parquet(split_dir / "X_val_encoded_scaled_final.parquet")
y_val_df = pd.read_parquet(split_dir / "y_val.parquet")

X_test_df = pd.read_parquet(split_dir / "X_test_encoded_scaled_final.parquet")
y_test_df = pd.read_parquet(split_dir / "y_test.parquet")

# Build DataLoaders
train_dataset = VehicleDataset(X_train_df, y_train_df)
val_dataset = VehicleDataset(X_val_df, y_val_df)
test_dataset = VehicleDataset(X_test_df, y_test_df)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [5]:
# Neural Network Architecture
# Implements a Multi-Layer Perceptron (MLP) regressor
class MLPRegressor(nn.Module):
    def __init__(self, input_dim: int):
        super(MLPRegressor, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128), #Normalizes the 128 feature outputs across the batch (zero mean, unit variance). 
            #Helps stabilize gradient flow and speeds up training.
            nn.GELU(), #GELU provides a smooth, probabilistic curvature. This allows the network to learn nuanced non-linear 
            #relationships across the features, which directly contributes to capturing 93.75% of the target variance.
            nn.Dropout(0.05),  # is a regularization technique in PyTorch designed to prevent the neural network from
            #overfitting (memorizing the training data).

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Dropout(0.05),

            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.GELU(),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)


# --- Instantiation & Optimization Setup ---
model = MLPRegressor(input_dim=len(SELECTED_FEATURES)).to(DEVICE)

# MODIFICATION 2: Switch to MSELoss to directly optimize for minimum squared error / RMSE
criterion = nn.MSELoss()

# MODIFICATION 3: Slightly reduced weight_decay (1e-4 -> 1e-5) to give network flexibility on tail values
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5)

In [6]:
# Training and Validation Loop with Early Stopping

best_val_loss = float('inf')
patience_counter = 0

for epoch in range(1, EPOCHS + 1):
    # Training
    model.train()
    train_loss = 0.0
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
        
        optimizer.zero_grad()
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * batch_X.size(0)
    
    train_loss /= len(train_loader.dataset)

    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(DEVICE), batch_y.to(DEVICE)
            predictions = model(batch_X)
            loss = criterion(predictions, batch_y)
            val_loss += loss.item() * batch_X.size(0)
            
    val_loss /= len(val_loader.dataset)
    scheduler.step(val_loss)

    if epoch % 5 == 0 or val_loss < best_val_loss:
        print(f"Epoch {epoch:03d} | Train Loss (MSE): {train_loss:.4f} | Val Loss (MSE): {val_loss:.4f}")

    # Checkpoint & Early Stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_mlp_model.pt')
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping triggered at epoch {epoch}.")
            break

Epoch 001 | Train Loss (MSE): 18682.7930 | Val Loss (MSE): 18280.0278
Epoch 002 | Train Loss (MSE): 18443.7206 | Val Loss (MSE): 18011.5429
Epoch 003 | Train Loss (MSE): 18169.7130 | Val Loss (MSE): 17678.0349
Epoch 004 | Train Loss (MSE): 17842.3966 | Val Loss (MSE): 17334.0495
Epoch 005 | Train Loss (MSE): 17455.3212 | Val Loss (MSE): 16979.3770
Epoch 006 | Train Loss (MSE): 17008.4925 | Val Loss (MSE): 16542.2930
Epoch 007 | Train Loss (MSE): 16508.0725 | Val Loss (MSE): 15938.7352
Epoch 008 | Train Loss (MSE): 15953.6733 | Val Loss (MSE): 15440.8102
Epoch 009 | Train Loss (MSE): 15353.6071 | Val Loss (MSE): 14723.1784
Epoch 010 | Train Loss (MSE): 14698.0564 | Val Loss (MSE): 14069.9534
Epoch 011 | Train Loss (MSE): 13986.1958 | Val Loss (MSE): 13465.5627
Epoch 012 | Train Loss (MSE): 13274.0471 | Val Loss (MSE): 12637.3069
Epoch 013 | Train Loss (MSE): 12550.5186 | Val Loss (MSE): 11955.3757
Epoch 014 | Train Loss (MSE): 11823.7763 | Val Loss (MSE): 11503.5680
Epoch 015 | Train Lo

In [7]:
#  Model Evaluation Strategy

def evaluate_model(model, loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch_X, batch_y in loader:
            batch_X = batch_X.to(DEVICE)
            out = model(batch_X)
            preds.extend(out.cpu().numpy().flatten())
            targets.extend(batch_y.numpy().flatten())
            
    preds, targets = np.array(preds), np.array(targets)
    mae = mean_absolute_error(targets, preds)
    rmse = np.sqrt(mean_squared_error(targets, preds))
    r2 = r2_score(targets, preds)
    return mae, rmse, r2

# Load best checkpoint
model.load_state_dict(torch.load('best_mlp_model.pt'))

# Overall Test Evaluation[cite: 5]
mae, rmse, r2 = evaluate_model(model, test_loader)
print(f"\n--- 2025 Test Set Metrics ---")
print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²:   {r2:.4f}")


--- 2025 Test Set Metrics ---
MAE:  9.9596
RMSE: 15.7691
R²:   0.9166


* **Average Deviation (±6.58 g/km):** On average, for any given vehicle, the model's CO₂ estimate differs from its actual official emission value by approximately 6.58 g/km. If a vehicle's actual emissions are 120.0 g/km, an average-sized prediction error would correspond to a prediction of approximately 113.4 g/km to 126.6 g/km.

* **Relative Error (approximately 6.3%):** An MAE of 6.5780 g/km represents approximately 6.3% of the average actual CO₂ emission value, assuming an average target value of approximately 104.84 g/km.